In [0]:
from pyspark.sql.functions import col, regexp_replace, to_timestamp

# 1. Read your Bronze table
bronze_df = spark.read.table("workspace.ais_db.bronze_ais_table")

# 2. Transform: Flatten, Cast Types, and Clean
silver_df = (
    bronze_df
    # Reach inside the JSON using dot notation and rename the columns nicely
    .select(
        col("MetaData.MMSI").cast("long").alias("mmsi"),
        col("MetaData.ShipName").alias("ship_name"),
        to_timestamp(
            regexp_replace(col("MetaData.time_utc"), " \\+0000 UTC$", ""),
            "yyyy-MM-dd HH:mm:ss.SSSSSSSSS"
        ).alias("timestamp_utc"),
        col("Message.PositionReport.Latitude").cast("double").alias("latitude"),
        col("Message.PositionReport.Longitude").cast("double").alias("longitude"),
        col("Message.PositionReport.Sog").cast("double").alias("speed_knots") # Speed Over Ground
    )
    # 3. Filter: Only keep rows that actually have GPS coordinates and a ship name
    .filter(col("latitude").isNotNull() & col("longitude").isNotNull() & col("ship_name").isNotNull())
)

# 4. Write to the Silver Layer
(silver_df.write
    .format("delta")
    .mode("overwrite") # We use overwrite here for batch transformations
    .saveAsTable("workspace.ais_db.silver_ais_table")
)

print("✅ Flattened and Cleaned!")
# display(spark.read.table("workspace.ais_db.silver_ais_table"))

In [0]:
display(spark.read.table("workspace.ais_db.silver_ais_table"))